In [ ]:
import os
import pandas as pd
import psycopg2
from psycopg2 import sql

# ---------- CONFIGURATION ----------
CSV_FOLDER = r"C:\Users\leedf\weatherData"  # Folder containing the CSV files
DB_NAME = "weather"
DB_USER = "postgres"
DB_PASSWORD = "071726postGRES"
DB_HOST = "localhost"
DB_PORT = "5432"

# ---------- CONNECT TO POSTGRES ----------
try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = True
    cursor = conn.cursor()
    print("Connected to PostgreSQL.")
except Exception as e:
    raise SystemExit(f"Database connection failed: {e}")

# ---------- PROCESS EACH CSV ----------
for file in os.listdir(CSV_FOLDER):
    if file.endswith(".csv"):
        table_name = os.path.splitext(file)[0]  # Table name from file name
        file_path = os.path.join(CSV_FOLDER, file)

        # Load CSV into DataFrame
        df = pd.read_csv(file_path)

        # Validate required columns
        required_cols = {"zip", "state", "city"}
        if not required_cols.issubset(df.columns):
            print(f"Skipping {file}: Missing required columns.")
            continue

        # Create table SQL dynamically
        col_defs = []
        for col in df.columns:
            if col == "zip":
                col_defs.append(sql.SQL("{} INTEGER PRIMARY KEY").format(sql.Identifier(col)))
            else:
                col_defs.append(sql.SQL("{} TEXT").format(sql.Identifier(col)))

        create_table_query = sql.SQL("CREATE TABLE IF NOT EXISTS {} ({});").format(
            sql.Identifier(table_name),
            sql.SQL(", ").join(col_defs)
        )

        try:
            cursor.execute(create_table_query)
            print(f"Table '{table_name}' created.")
        except Exception as e:
            print(f"Error creating table {table_name}: {e}")
            continue

        # Insert data
        for _, row in df.iterrows():
            insert_query = sql.SQL("INSERT INTO {} ({}) VALUES ({}) ON CONFLICT (zip) DO NOTHING;").format(
                sql.Identifier(table_name),
                sql.SQL(", ").join(map(sql.Identifier, df.columns)),
                sql.SQL(", ").join(sql.Placeholder() * len(df.columns))
            )
            try:
                cursor.execute(insert_query, tuple(row))
            except Exception as e:
                print(f"Insert error in {table_name}: {e}")

print("All CSV files processed.")

# ---------- CLEANUP ----------
cursor.close()
conn.close()


Connected to PostgreSQL.
Table 'us_average_humidity_by_zip' created.
Table 'us_average_precipitation_by_zip' created.
Table 'us_average_snowfall_by_zip' created.
Table 'us_average_temperature_by_zip' created.
All CSV files processed.


In [4]:
import pandas as pd
wind_data = pd.read_csv(r"C:\Users\leedf\Downloads\annual_avg_windspeed_data.csv")
wind_data.head()


,zip_code,latitude,longitude,station_id,annual_avg_awnd_mph
0,98253,48.098624,-122.580049,GHCND:USW00024255,No AWND data for year
1,98271,48.118345,-122.171071,GHCND:USW00024255,No AWND data for year
2,98324,48.091757,-123.172486,GHCND:USW00094276,4.841630901287553
3,99148,48.082743,-117.620107,GHCND:USW00024157,7.996952908587265
4,98834,48.140121,-120.019759,GHCND:USC00451350,2.0835654596100213


In [5]:
import os
import pandas as pd
import psycopg2
from psycopg2 import sql

# ---------- CONFIGURATION ----------
CSV_FOLDER = r"C:\Users\leedf\wind_speed"  # Folder containing the CSV files
DB_NAME = "weather"
DB_USER = "postgres"
DB_PASSWORD = "071726postGRES"
DB_HOST = "localhost"
DB_PORT = "5432"

# ---------- CONNECT TO POSTGRES ----------
try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    conn.autocommit = True
    cursor = conn.cursor()
    print("Connected to PostgreSQL.")
except Exception as e:
    raise SystemExit(f"Database connection failed: {e}")

# ---------- PROCESS EACH CSV ----------
for file in os.listdir(CSV_FOLDER):
    if file.endswith(".csv"):
        table_name = os.path.splitext(file)[0]  # Table name from file name
        file_path = os.path.join(CSV_FOLDER, file)

        # Load CSV into DataFrame
        df = pd.read_csv(file_path)

        # Validate required columns
        required_cols = {"zip_code", "latitude", "longitude", "station_id"}
        if not required_cols.issubset(df.columns):
            print(f"Skipping {file}: Missing required columns.")
            continue

        # Create table SQL dynamically
        col_defs = []
        for col in df.columns:
            if col == "zip_code":
                col_defs.append(sql.SQL("{} INTEGER PRIMARY KEY").format(sql.Identifier(col)))
            else:
                col_defs.append(sql.SQL("{} TEXT").format(sql.Identifier(col)))

        create_table_query = sql.SQL("CREATE TABLE IF NOT EXISTS {} ({});").format(
            sql.Identifier(table_name),
            sql.SQL(", ").join(col_defs)
        )

        try:
            cursor.execute(create_table_query)
            print(f"Table '{table_name}' created.")
        except Exception as e:
            print(f"Error creating table {table_name}: {e}")
            continue

        # Insert data
        for _, row in df.iterrows():
            insert_query = sql.SQL("INSERT INTO {} ({}) VALUES ({}) ON CONFLICT (zip_code) DO NOTHING;").format(
                sql.Identifier(table_name),
                sql.SQL(", ").join(map(sql.Identifier, df.columns)),
                sql.SQL(", ").join(sql.Placeholder() * len(df.columns))
            )
            try:
                cursor.execute(insert_query, tuple(row))
            except Exception as e:
                print(f"Insert error in {table_name}: {e}")

print("All CSV files processed.")

# ---------- CLEANUP ----------
cursor.close()
conn.close()


Connected to PostgreSQL.
Table 'annual_avg_windspeed_data' created.
All CSV files processed.


In [9]:
# Create a pandas dataframe from merged tables in a PostgreSQL database
import pandas as pd
from sqlalchemy import create_engine

# ---------- CONFIGURATION ----------
DB_NAME = "weather"
DB_USER = "postgres"
DB_PASSWORD = "071726postGRES"
DB_HOST = "localhost"
DB_PORT = "5432"

# ---------- CONNECT TO POSTGRES ----------
connection_string = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string)

# ---------- ALTER TABLES DATATYPES ----------
query = '''
SELECT 
    h.*, 
    p.avg_monthly_precipitation_in AS precipitation, 
    s.avg_monthly_snowfall_in AS snowfall, 
    t.avg_temperature_f AS temperature , 
    w.annual_avg_awnd_mph AS wind 
FROM us_average_humidity_by_zip AS h 
INNER JOIN us_average_precipitation_by_zip AS p ON h.zip = p.zip
INNER JOIN us_average_snowfall_by_zip AS s ON h.zip = s.zip
INNER JOIN us_average_temperature_by_zip AS t ON h.zip = t.zip
INNER JOIN annual_avg_windspeed_data AS w ON h.zip = w.zip_code;
'''

# Read the joined query into a pandas dataframe
df = pd.read_sql(query, con=engine)

# Display the first few rows of the dataframe
df.head()

,zip,city,state,avg_relative_humidity_pct,precipitation,snowfall,temperature,wind
0,98253,Greenbank,WA,75.294738705695,2.50803943624694,0.3562992125984252,51.58910360940877,No AWND data for year
1,98271,Marysville,WA,78.26610605831172,2.50803943624694,0.3562992125984252,50.7394606643558,No AWND data for year
2,98324,Carlsborg,WA,78.83734773630276,2.275196850393701,0.7027559055118111,49.64409138071476,4.841630901287553
3,99148,Loon Lake,WA,66.87719089337939,1.7293307086614174,3.762139107611549,47.096120338678034,7.996952908587265
4,98834,Methow,WA,59.05169803067157,1.2137795275590553,3.768747656542932,51.04268979565974,2.0835654596100213


In [ ]:
# Save the dataframe to a CSV file
df.to_csv(r"C:\Users\leedf\weatherData\weather_data_v1.csv", index=False)